# 原始特征 XGBoost：控制变量的起点

仅使用原始 26 个输入字段；这是后续特征工程的对照组。

学习目标：Boosting、原生类别、早停、固定验证方案。

先按顺序运行，再修改一个因素。历史分数与本次运行分数分开记录。

## 1. 导入与路径
本课只导入本文件夹的模型和包内的数据工具。

In [ ]:
from pathlib import Path
import sys
import pickle
import numpy as np
from sklearn.model_selection import KFold
HERE = Path.cwd()
if not (HERE / 'model.py').exists():
    raise RuntimeError('请从当前课程文件夹打开这个 Notebook。')
sys.path.insert(0, str(HERE.parent / '_shared'))
from course_utils import DATA_DIR, load_data, rmsle, save_outputs
from model import make_features, fit_model, predict_model, best_rounds

data_dir = HERE.parent/'data'
output_dir = HERE/'outputs'


## 2. 读取数据与目标
任务是预测连续数值 result，id 只用于对齐结果。

In [ ]:

train, test = load_data(data_dir)
X = train.drop(columns=['id', 'result'])
T = test.drop(columns=['id'])
target = np.log1p(train.result.to_numpy())
print('训练集 / 测试集：', train.shape, test.shape)


In [ ]:
display(train.head())
display(train['result'].describe())
# 仅展示特征形状；这个演示处理器不参与下面的验证训练。
example = make_features()
example_matrix = example.fit_transform(X.iloc[:1000], target[:1000])
print('模型输入列数：', len(example_matrix.columns))
print(example_matrix.columns.tolist())


## 3. 五折验证
先读 model.py。特征拟合和模型训练只能发生在训练折内。

In [ ]:

folds = list(KFold(n_splits=5, shuffle=True, random_state=42).split(X))
oof = np.zeros(len(train))
test_mean = np.zeros(len(test))
fold_scores, round_counts, estimators = [], [], []
for fold, (train_index, valid_index) in enumerate(folds):
    # 只在训练折拟合特征；验证折只 transform，不能泄漏标签。
    features = make_features()
    A = features.fit_transform(X.iloc[train_index], target[train_index])
    B = features.transform(X.iloc[valid_index])
    estimator = fit_model(A, target[train_index], B, target[valid_index])
    valid_prediction = np.clip(np.expm1(predict_model(estimator, B)), 0, 1)
    oof[valid_index] = valid_prediction
    score = rmsle(train.result.iloc[valid_index], valid_prediction)
    fold_scores.append(score)
    round_counts.append(best_rounds(estimator))
    print(f'Fold {fold + 1}: RMSLE={score:.6f}')



## 4. 统一计算 OOF
合并所有验证行再算 RMSLE，不能拿训练分数代替。

In [ ]:

score = rmsle(train.result, oof)
print(f'完整 OOF RMSLE: {score:.6f}')


## 5. 生成最终预测
历史提交由全量训练的一个模型生成；五折模型只用于 OOF 和选轮数。

In [ ]:
# 用验证选轮数；实际提交由一个全量训练模型产生。
final_rounds = int(np.median(round_counts))
features = make_features()
A = features.fit_transform(X, target)
estimator = fit_model(A, target, rounds=final_rounds)
predictions = np.clip(np.expm1(predict_model(estimator, features.transform(T))), 0, 1)
estimators = [(features, estimator)]
prediction_rule = 'single_full'
print('全量训练轮数：', final_rounds)


## 6. 保存本次结果

In [ ]:

metrics = {'oof_rmsle': score, 'fold_scores': fold_scores,
           'validation': 'KFold(5, shuffle=True, random_state=42)',
           'prediction_rule': prediction_rule, 'final_rounds': final_rounds}
save_outputs(output_dir, train, test, oof, predictions, metrics)
with (Path(output_dir) / 'checkpoint.pkl').open('wb') as file:
    pickle.dump({'prediction_rule': prediction_rule, 'estimators': estimators}, file)
print('已保存到：', Path(output_dir).resolve())


## 7. 阅读历史线上成绩
下面读的是已提交记录，不是本次运行的线上成绩。

In [ ]:
import json
record = json.loads((HERE/'reference/scores.json').read_text())
record

## 练习与讨论
保持划分和特征不变，只改变树深度，判断算法参数带来的真实增益。